# Day 9 v3 — Silver Blob Transformation: Job Parameters + Data Quality

**Source:** `/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/<entity>/YYYY/MM/DD/HH/`
**Sink:** `/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime/<entity>/` (Delta)

**Production version — attaches to existing Databricks Job `job_bronze_realtime_hourly`**

**What v3 adds over v2:**
| | v2 | v3 |
|---|---|---|
| Parameters | Hardcoded | Job parameters only — raises error if not provided |
| Load mode | Full overwrite only | `full` or `incremental` (Delta MERGE) |
| Date filter | All partitions | Specific `YYYY/MM/DD/HH` partition from job |
| Data quality | None | Null PK, null CDC, corrupt cast, negative values → quarantine |
| Run summary | Basic | Per-entity quality breakdown table |

**Job parameter values passed by Databricks Job:**
```
load_type        : incremental
ingestion_year   : 2026
ingestion_month  : 07
ingestion_day    : 15
ingestion_hour   : 06     (blank = all hours for that day)
```

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Read parameters from Databricks Job ONLY
# Parameters must be injected by the job — notebook raises ValueError if missing.

def _get_job_param(key):
    try:
        value = dbutils.widgets.get(key).strip()
    except Exception:
        raise ValueError(
            f"Parameter '{key}' was not provided. "
            f"This notebook must be run via a Databricks Job — not interactively."
        )
    if not value:
        raise ValueError(f"Parameter '{key}' is empty. Set it in the Databricks Job task parameters.")
    return value


LOAD_TYPE       = _get_job_param("load_type")
INGESTION_YEAR  = _get_job_param("ingestion_year")
INGESTION_MONTH = _get_job_param("ingestion_month")
INGESTION_DAY   = _get_job_param("ingestion_day")

# Hour is optional — blank means process all hours for the given day
try:
    INGESTION_HOUR = dbutils.widgets.get("ingestion_hour").strip()
except Exception:
    INGESTION_HOUR = ""

if LOAD_TYPE not in ("full", "incremental"):
    raise ValueError(f"load_type must be 'full' or 'incremental', got: {LOAD_TYPE!r}")

print(f"load_type       : {LOAD_TYPE}")
print(f"ingestion_year  : {INGESTION_YEAR}")
print(f"ingestion_month : {INGESTION_MONTH}")
print(f"ingestion_day   : {INGESTION_DAY}")
print(f"ingestion_hour  : {INGESTION_HOUR or '(all hours)'}")
print("Parameters validated — continuing.")

In [ ]:
# Cell 3 — Constants
BRONZE_REALTIME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime"
SILVER_REALTIME = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime"
QUARANTINE_BASE = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/quarantine/realtime"

PIPELINE_NAME = "job_silver_blob_realtime_v3"
RUN_TS        = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"BRONZE_REALTIME : {BRONZE_REALTIME}")
print(f"SILVER_REALTIME : {SILVER_REALTIME}")
print(f"QUARANTINE_BASE : {QUARANTINE_BASE}")
print(f"RUN_TS          : {RUN_TS}")

In [ ]:
# Cell 4 — Entity schema registry
# cast maps use only the real CSV column names from Bronze.
# updated_at is NOT in the cast map — it is derived from the file path in step_read_csv().

ENTITY_REGISTRY = {
    "charging_sessions": {
        "natural_key": "session_id",
        "cdc_field"  : "updated_at",
        "cast": {
            "session_id"    : "string",
            "charger_id"    : "string",
            "vehicle_id"    : "string",
            "station_id"    : "string",
            "customer_id"   : "string",
            "plug_in_ts"    : "timestamp",
            "charge_end_ts" : "timestamp",
            "energy_kwh"    : "decimal(10,4)",
            "session_status": "string",
            "tariff_id"     : "string",
        }
    },
    "maintenance_events": {
        "natural_key": "event_id",
        "cdc_field"  : "updated_at",
        "cast": {
            "event_id"      : "string",
            "charger_id"    : "string",
            "station_id"    : "string",
            "event_type"    : "string",
            "description"   : "string",
            "technician_id" : "string",
            "event_status"  : "string",
            "scheduled_date": "date",
            "completed_date": "date",
        }
    },
}

NEGATIVE_CHECK_COLS = {
    "charging_sessions"  : ["energy_kwh"],
    "maintenance_events" : [],
}

print(f"Entity registry loaded: {len(ENTITY_REGISTRY)} entities")

In [ ]:
# Cell 5 — Helper functions

def get_bronze_paths(entity_name):
    """Return CSV file paths for the given entity and partition parameters.
    Full load -> all partitions. Incremental -> specific YYYY/MM/DD/HH partition."""
    base = f"{BRONZE_REALTIME}/{entity_name}"
    try:
        dbutils.fs.ls(base)
    except Exception:
        return []

    if LOAD_TYPE == "full":
        return [base]  # spark.read.csv reads recursively

    # Incremental — build partition path from job parameters
    if INGESTION_HOUR:
        partition_path = f"{base}/{INGESTION_YEAR}/{INGESTION_MONTH}/{INGESTION_DAY}/{INGESTION_HOUR}"
    else:
        partition_path = f"{base}/{INGESTION_YEAR}/{INGESTION_MONTH}/{INGESTION_DAY}"

    try:
        dbutils.fs.ls(partition_path)
        return [partition_path]
    except Exception:
        return []


def folder_exists_dbfs(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def write_quarantine(df, entity_name, reason):
    if df.limit(1).count() == 0:
        return
    (
        df.withColumn("reject_reason", F.lit(reason))
          .withColumn("reject_ts",     F.lit(RUN_TS).cast("timestamp"))
          .write.format("delta")
          .mode("append")
          .option("mergeSchema", "true")
          .save(f"{QUARANTINE_BASE}/{entity_name}")
    )


print("Helpers defined")

In [ ]:
# Cell 6 — Step 1: Read Bronze CSV + derive updated_at from file path
# updated_at is not a column in the CSV — it is the partition timestamp (YYYY/MM/DD/HH)
# extracted from _metadata.file_path. This is the CDC field used for deduplication.

def step_read_csv(paths):
    raw_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("recursiveFileLookup", "true")
        .csv(*paths)
        .select("*", "_metadata.file_path")
    )

    raw_df = (
        raw_df
        .withColumn("_year",  F.regexp_extract(F.col("file_path"), r"/(\d{4})/", 1))
        .withColumn("_month", F.regexp_extract(F.col("file_path"), r"/\d{4}/(\d{2})/", 1))
        .withColumn("_day",   F.regexp_extract(F.col("file_path"), r"/\d{4}/\d{2}/(\d{2})/", 1))
        .withColumn("_hour",  F.regexp_extract(F.col("file_path"), r"/\d{4}/\d{2}/\d{2}/(\d{2})/", 1))
        .withColumn(
            "updated_at",
            F.to_timestamp(
                F.concat_ws(" ",
                    F.concat_ws("-", F.col("_year"), F.col("_month"), F.col("_day")),
                    F.col("_hour")
                ),
                "yyyy-MM-dd HH"
            )
        )
        .drop("file_path", "_year", "_month", "_day", "_hour")
    )
    return raw_df

print("step_read_csv() defined")

In [ ]:
# Cell 7 — Step 2: Trim whitespace from all string columns

def step_trim_strings(df):
    string_cols = [c for c, t in df.dtypes if t == "string"]
    for col in string_cols:
        df = df.withColumn(col, F.trim(F.col(col)))
    return df

print("step_trim_strings() defined")

In [ ]:
# Cell 8 — Step 3: Drop rows with NULL or empty primary key

def step_filter_null_pk(df, entity_name, natural_key):
    null_pk_df = df.filter(F.col(natural_key).isNull() | (F.col(natural_key) == ""))
    count = null_pk_df.count()
    if count > 0:
        write_quarantine(null_pk_df, entity_name, f"null_primary_key:{natural_key}")
    clean_df = df.filter(F.col(natural_key).isNotNull() & (F.col(natural_key) != ""))
    return clean_df, count

print("step_filter_null_pk() defined")

In [ ]:
# Cell 9 — Step 4: Drop rows with NULL CDC timestamp

def step_filter_null_cdc(df, entity_name, cdc_field):
    null_cdc_df = df.filter(F.col(cdc_field).isNull() | (F.col(cdc_field) == ""))
    count = null_cdc_df.count()
    if count > 0:
        write_quarantine(null_cdc_df, entity_name, f"null_cdc_field:{cdc_field}")
    clean_df = df.filter(F.col(cdc_field).isNotNull() & (F.col(cdc_field) != ""))
    return clean_df, count

print("step_filter_null_cdc() defined")

In [ ]:
# Cell 10 — Step 5: Cast columns + detect corrupt rows
# Pre-cast sentinel: only flag as corrupt if value existed before cast but became NULL after.
# updated_at is derived from the file path (already a timestamp) — pass it through unchanged.

def step_cast_and_check_corrupt(raw_df, entity_name, cast_map):
    numeric_types = ("decimal", "integer", "long", "double", "float")
    numeric_cols  = [
        c for c, t in cast_map.items()
        if any(t.startswith(nt) for nt in numeric_types)
        and c in raw_df.columns
    ]

    # Add pre-cast sentinel flags for each numeric column
    for c in numeric_cols:
        flag_col = f"_pre_{c}"
        raw_df = raw_df.withColumn(
            flag_col,
            F.col(c).isNotNull() & (F.col(c).cast("string") != "")
        )

    # Build cast expressions — only for columns in the registry
    cast_exprs = []
    for col_name, col_type in cast_map.items():
        if col_name in raw_df.columns:
            cast_exprs.append(F.col(col_name).cast(col_type).alias(col_name))
        else:
            cast_exprs.append(F.lit(None).cast(col_type).alias(col_name))

    # Preserve updated_at (derived from file path, not in cast_map)
    cast_exprs.append(F.col("updated_at").cast("timestamp"))

    # Preserve pre-cast sentinel flags
    flag_cols  = [f"_pre_{c}" for c in numeric_cols]
    flag_exprs = [F.col(f) for f in flag_cols]

    # Preserve any other columns not in cast_map (e.g. extra CSV fields)
    registry_cols = set(cast_map.keys()) | {"updated_at"} | set(flag_cols)
    extra_exprs   = [F.col(c) for c in raw_df.columns if c not in registry_cols]

    typed_df = raw_df.select(cast_exprs + extra_exprs + flag_exprs)

    # Detect and quarantine corrupt rows
    corrupt_count = 0
    if numeric_cols:
        corrupt_condition = F.lit(False)
        for c in numeric_cols:
            flag = f"_pre_{c}"
            corrupt_condition = corrupt_condition | (F.col(flag) & F.col(c).isNull())

        corrupt_df    = typed_df.filter(corrupt_condition)
        corrupt_count = corrupt_df.drop(*flag_cols).count()
        if corrupt_count > 0:
            write_quarantine(corrupt_df.drop(*flag_cols), entity_name, "corrupt_cast_to_null")
        typed_df = typed_df.filter(~corrupt_condition)

    typed_df = typed_df.drop(*flag_cols)
    return typed_df, corrupt_count

print("step_cast_and_check_corrupt() defined")

In [ ]:
# Cell 11 — Step 6: Quarantine rows with negative numeric values

def step_filter_negative_values(df, entity_name):
    neg_cols = [c for c in NEGATIVE_CHECK_COLS.get(entity_name, []) if c in df.columns]
    if not neg_cols:
        return df, 0

    negative_condition = F.lit(False)
    for c in neg_cols:
        negative_condition = negative_condition | (F.col(c) < 0)

    negative_df = df.filter(negative_condition)
    count       = negative_df.count()
    if count > 0:
        write_quarantine(negative_df, entity_name, "negative_numeric_value")
    clean_df = df.filter(~negative_condition)
    return clean_df, count

print("step_filter_negative_values() defined")

In [ ]:
# Cell 12 — Step 7: Add Silver audit columns + Deduplicate

def step_add_audit_and_dedup(df, natural_key, cdc_field):
    df = (
        df
        .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
        .withColumn("silver_load_type",   F.lit(LOAD_TYPE))
        .withColumn("silver_pipeline",    F.lit(PIPELINE_NAME))
    )

    before_count = df.count()
    window = Window.partitionBy(natural_key).orderBy(F.col(cdc_field).desc())
    deduped_df = (
        df
        .withColumn("_row_num", F.row_number().over(window))
        .filter(F.col("_row_num") == 1)
        .drop("_row_num")
    )
    dup_count = before_count - deduped_df.count()
    return deduped_df, dup_count

print("step_add_audit_and_dedup() defined")

In [ ]:
# Cell 13 — Step 8: Write to Silver Delta
# Full load  -> overwrite entire Silver table.
# Incremental -> Delta MERGE upsert on natural key.

def step_write_silver(df, entity_name, natural_key, silver_path):
    writer = df.write.format("delta").option("mergeSchema", "true")

    if LOAD_TYPE == "full" or not folder_exists_dbfs(silver_path):
        writer.mode("overwrite").save(silver_path)
    else:
        if DeltaTable.isDeltaTable(spark, silver_path):
            delta_table = DeltaTable.forPath(spark, silver_path)
            (
                delta_table.alias("target")
                .merge(
                    df.alias("source"),
                    f"target.{natural_key} = source.{natural_key}"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
        else:
            writer.mode("overwrite").save(silver_path)

print("step_write_silver() defined")

In [ ]:
# Cell 14 — Orchestrator: transform_entity()

def transform_entity(entity_name, config):
    result = {
        "entity_name"        : entity_name,
        "status"             : "failed",
        "bronze_rows"        : 0,
        "null_pk_dropped"    : 0,
        "null_cdc_dropped"   : 0,
        "corrupt_quarantine" : 0,
        "negative_quarantine": 0,
        "duplicate_dropped"  : 0,
        "silver_rows"        : 0,
        "error"              : None,
    }
    try:
        natural_key = config["natural_key"]
        cdc_field   = config["cdc_field"]
        cast_map    = config["cast"]
        silver_path = f"{SILVER_REALTIME}/{entity_name}"

        paths = get_bronze_paths(entity_name)
        if not paths:
            result["status"] = "skipped"
            result["error"]  = "No Bronze CSV files found for given partition"
            return result

        # Step 1 — Read Bronze CSV
        raw_df = step_read_csv(paths)
        result["bronze_rows"] = raw_df.count()

        # Step 2 — Trim whitespace
        raw_df = step_trim_strings(raw_df)

        # Step 3 — Drop null primary key
        raw_df, result["null_pk_dropped"] = step_filter_null_pk(raw_df, entity_name, natural_key)

        # Step 4 — Drop null CDC field
        raw_df, result["null_cdc_dropped"] = step_filter_null_cdc(raw_df, entity_name, cdc_field)

        # Step 5 — Cast + corrupt check
        typed_df, result["corrupt_quarantine"] = step_cast_and_check_corrupt(raw_df, entity_name, cast_map)

        # Step 6 — Negative value check
        typed_df, result["negative_quarantine"] = step_filter_negative_values(typed_df, entity_name)

        # Step 7 — Audit columns + dedup
        deduped_df, result["duplicate_dropped"] = step_add_audit_and_dedup(typed_df, natural_key, cdc_field)

        # Step 8 — Write to Silver
        step_write_silver(deduped_df, entity_name, natural_key, silver_path)

        result["silver_rows"] = deduped_df.count()
        result["status"]      = "succeeded"

    except Exception as e:
        result["error"] = str(e)

    return result

print("transform_entity() orchestrator defined")

In [ ]:
# Cell 15 — Run all entities

partition_label = f"{INGESTION_YEAR}/{INGESTION_MONTH}/{INGESTION_DAY}" + (f"/{INGESTION_HOUR}" if INGESTION_HOUR else "")
print(f"Starting Silver blob transformation — load_type={LOAD_TYPE} | partition={partition_label}")
print("-" * 65)

results = []
for entity_name, config in ENTITY_REGISTRY.items():
    print(f"  Processing: {entity_name} ...", end=" ")
    r = transform_entity(entity_name, config)
    results.append(r)
    if r["status"] == "succeeded":
        print(f"OK  (bronze={r['bronze_rows']} -> silver={r['silver_rows']})")
    elif r["status"] == "skipped":
        print(f"SKIP  ({r['error']})")
    else:
        print(f"FAILED  ({r['error']})")

print("-" * 65)
print("All entities processed")

In [ ]:
# Cell 16 — Run summary

succeeded = [r for r in results if r["status"] == "succeeded"]
skipped   = [r for r in results if r["status"] == "skipped"]
failed    = [r for r in results if r["status"] == "failed"]

print("=" * 95)
print("SILVER BLOB TRANSFORMATION v3 — RUN SUMMARY")
print("=" * 95)
print(f"  load_type  : {LOAD_TYPE}")
print(f"  partition  : {partition_label}")
print(f"  run_ts     : {RUN_TS}")
print(f"  succeeded  : {len(succeeded)}  |  skipped: {len(skipped)}  |  failed: {len(failed)}")
print()
print(f"  {'ENTITY':<25} {'STATUS':<10} {'BRONZE':>8} {'NULL_PK':>8} {'NULL_CDC':>9} {'CORRUPT':>8} {'NEGATIVE':>9} {'DUPS':>6} {'SILVER':>8}")
print(f"  {'-'*25} {'-'*10} {'-'*8} {'-'*8} {'-'*9} {'-'*8} {'-'*9} {'-'*6} {'-'*8}")

for r in results:
    tag = "[OK]  " if r["status"] == "succeeded" else ("[SKIP]" if r["status"] == "skipped" else "[FAIL]")
    print(
        f"  {tag} {r['entity_name']:<25} {r['status']:<10}"
        f" {r['bronze_rows']:>8}"
        f" {r['null_pk_dropped']:>8}"
        f" {r['null_cdc_dropped']:>9}"
        f" {r['corrupt_quarantine']:>8}"
        f" {r['negative_quarantine']:>9}"
        f" {r['duplicate_dropped']:>6}"
        f" {r['silver_rows']:>8}"
    )
    if r["error"]:
        print(f"         Error: {r['error']}")

print()
total_q = sum(r["null_pk_dropped"] + r["null_cdc_dropped"] + r["corrupt_quarantine"] + r["negative_quarantine"] for r in results)
print(f"  Total quarantined : {total_q}")
print(f"  Total in Silver   : {sum(r['silver_rows'] for r in results)}")

if failed:
    raise Exception(f"{len(failed)} entity(ies) failed — check output above.")
else:
    print(f"\nSilver blob transformation complete.")

In [ ]:
# Cell 17 — Verify Silver + Quarantine

print("SILVER VERIFICATION")
print("-" * 50)
for entity_name in ENTITY_REGISTRY:
    path = f"{SILVER_REALTIME}/{entity_name}"
    if not folder_exists_dbfs(path):
        print(f"  {entity_name}: NOT FOUND")
        continue
    df = spark.read.format("delta").load(path)
    print(f"  {entity_name:<25} rows={df.count():>8}  cols={len(df.columns)}")

print()
print("QUARANTINE CHECK")
print("-" * 50)
for entity_name in ENTITY_REGISTRY:
    q_path = f"{QUARANTINE_BASE}/{entity_name}"
    if not folder_exists_dbfs(q_path):
        print(f"  {entity_name:<25} no quarantine rows")
        continue
    q_df = spark.read.format("delta").load(q_path)
    print(f"  {entity_name:<25} quarantine rows={q_df.count()}")
    q_df.groupBy("reject_reason").count().orderBy("count", ascending=False).show(truncate=False)